# 📦 01 — Synthetic Data Generation

**Responsibility:** Create synthetic datasets of electric vehicles (EV) and drivers,
with configurable separation levels, noise, and punctuality profiles.

**Main outputs:**
- `vehicles_extended.csv` — Base electric vehicle dataset
- `vehicles_precision_1.csv` … `vehicles_precision_6.csv` — Datasets with increasing separation
- `dataset_parametrizado.csv` — Custom dataset by groups (punctual/unpunctual)
- `dataset_final.csv` — Dataset with configurable noise for advanced experiments


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import random

## 2. Base Dataset Generator (original format)

Generates a dataset in the **original format** of the project with all the columns
needed for clustering: vehicle data, battery and driver profile.


In [ ]:
def generar_dataset_formato_original(n, guardar_csv=False, nombre_archivo="dataset_generado.csv"):
    """
    Generates a synthetic dataset of n vehicles/drivers in the
    original format of the project.

    Generated columns:
    - id: unique record identifier
    - distance: distance traveled since last charge (km)
    - available_time_start/end: availability time window for charging
    - current_charge: current battery charge (kWh)
    - battery_capacity: total battery capacity (kWh)
    - charge_speed: charging speed (kW)
    - discharge_rate: discharge rate (kW)
    - driver_age: driver age
    - driver_profession: profession (worker/student/retired)
    - use_frequency: days per week of usage
    - origin_distance: distance to the usual origin point (km)
    """

    # Available professions for drivers
    profesiones = ["worker", "student", "retired"]
    datos = []

    for i in range(1, n + 1):

        # --- Driver type selection with realistic weights ---
        # Workers are the most common group (50%), followed by students (30%)
        tipo = random.choices(profesiones, weights=[0.5, 0.3, 0.2])[0]

        # --- Age by profession ---
        # Each profession has a representative age range
        if tipo == "worker":
            edad = random.randint(25, 55)
        elif tipo == "student":
            edad = random.randint(18, 26)
        else:  # retired
            edad = random.randint(60, 85)

        # --- Availability hours ---
        # Workers have more predictable schedules (morning/afternoon)
        # Students and retirees are more variable
        def hora_random(h1, h2):
            return random.randint(h1, h2)

        if tipo == "worker":
            h_ini = hora_random(7, 9)
            h_fin = hora_random(15, 17)
        elif tipo == "student":
            h_ini = hora_random(8, 20)
            h_fin = hora_random(h_ini + 1, min(h_ini + 12, 23))
        else:  # retired
            h_ini = hora_random(16, 20)
            h_fin = hora_random(h_ini + 1, min(h_ini + 6, 23))

        # --- Distances by driver type ---
        # Retirees tend to have a more distant origin, reflecting lower usage frequency
        if tipo == "worker":
            origin_dist = max(1, int(np.random.normal(8, 3)))
            dist = max(10, int(np.random.normal(120, 40)))
        elif tipo == "student":
            origin_dist = max(1, int(np.random.normal(10, 4)))
            dist = max(10, int(np.random.normal(140, 50)))
        else:
            origin_dist = max(1, int(np.random.normal(20, 6)))
            dist = max(10, int(np.random.normal(180, 60)))

        # --- Usage frequency (days/week) ---
        if tipo == "worker":
            frecuencia = random.randint(4, 7)
        elif tipo == "student":
            frecuencia = random.randint(2, 5)
        else:
            frecuencia = random.randint(1, 3)

        # --- Vehicle electrical parameters ---
        battery_capacity = random.choice([80, 100, 120, 150])  # kWh
        current_charge = random.randint(0, battery_capacity)
        charge_speed = random.randint(5, 15)     # kW
        discharge_rate = random.randint(3, 10)   # kW

        datos.append({
            "id": i,
            "distance": dist,
            "available_time_start": h_ini,
            "available_time_end": h_fin,
            "current_charge": current_charge,
            "battery_capacity": battery_capacity,
            "charge_speed": charge_speed,
            "discharge_rate": discharge_rate,
            "driver_age": edad,
            "driver_profession": tipo,
            "use_frequency": frecuencia,
            "origin_distance": origin_dist
        })

    df = pd.DataFrame(datos)

    if guardar_csv:
        df.to_csv(nombre_archivo, index=False)
        print(f"✔ Archivo generado: {nombre_archivo}")

    return df


# --- Usage example: generate and save the base dataset ---
df_base = generar_dataset_formato_original(200, guardar_csv=True, nombre_archivo="vehicles_extended.csv")
print(df_base.head())


## 3. Datasets Generator with Increasing Separation

Creates **6 versions** of the base dataset with increasing separation between groups.
Used to evaluate the robustness of clustering.

- **Level 1**: lots of noise and very mixed groups (hard to separate)
- **Level 6**: very well-differentiated groups (easy to separate)


In [ ]:
def generar_datasets_precision(df_base):
    """
    Starting from the base dataset, generates 6 versions with increasing separation
    between punctual and unpunctual users.

    Strategy:
    1. Calculate the stress_index (ratio required_time / available_window)
    2. Label as 'unpunctual' those above the median
    3. For each level i: add decreasing noise and amplify differences
       for unpunctual users with an increasing separation factor

    Key parameters:
    - noise_levels: noise level per version (higher → harder)
    - separation_factors: stress_index multiplier for unpunctual users
    """

    df = df_base.copy()

    # --- Base feature engineering ---
    # Compute derived metrics needed for the stress_index
    df["battery_level"] = df["current_charge"] / df["battery_capacity"]
    df["window_length"] = df["available_time_end"] - df["available_time_start"]
    df["required_energy"] = df["battery_capacity"] - df["current_charge"]
    df["required_time"] = df["required_energy"] / df["charge_speed"]
    # If the window is 0, use 0.5h to avoid division by zero
    df["stress_index"] = df["required_time"] / df["window_length"].replace(0, 0.5)

    # --- Base label: unpunctual if stress_index > median ---
    median_stress = df["stress_index"].median()
    df["user_type"] = np.where(df["stress_index"] > median_stress, "unpunctual", "punctual")

    np.random.seed(42)

    # Level 1: lots of noise, low separation → groups are mixed
    # Level 6: low noise, high separation → groups are clearly distinct
    noise_levels = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
    separation_factors = [1.0, 1.5, 2.0, 3.0, 5.0, 10.0]

    for i in range(6):
        df_mod = df.copy()

        # Add Gaussian noise proportional to each column's standard deviation
        for col in ["current_charge", "charge_speed", "available_time_start", "available_time_end", "distance"]:
            scale = df_mod[col].std() * noise_levels[i]
            df_mod[col] += np.random.normal(0, scale, size=len(df_mod))

        # Recompute derived features after applying noise
        df_mod["battery_level"] = df_mod["current_charge"] / df_mod["battery_capacity"]
        df_mod["window_length"] = df_mod["available_time_end"] - df_mod["available_time_start"]
        df_mod["required_energy"] = df_mod["battery_capacity"] - df_mod["current_charge"]
        df_mod["required_time"] = df_mod["required_energy"] / df_mod["charge_speed"]
        df_mod["stress_index"] = df_mod["required_time"] / df_mod["window_length"].replace(0, 0.5)

        # Amplify unpunctual users' features to create
        # more separation between groups (more stress, shorter time window)
        factor = separation_factors[i]
        df_mod.loc[df_mod["user_type"] == "unpunctual", "stress_index"] *= factor
        df_mod.loc[df_mod["user_type"] == "unpunctual", "required_time"] *= factor
        df_mod.loc[df_mod["user_type"] == "unpunctual", "window_length"] /= factor
        df_mod.loc[df_mod["user_type"] == "unpunctual", "battery_level"] *= 0.9  # lower charge

        filename = f"vehicles_precision_{i+1}.csv"
        df_mod.to_csv(filename, index=False)
        print(f"✅ Guardado {filename} (nivel de precisión {i+1})")

    print("\nDatasets generados del nivel 1 (muy mezclados) al nivel 6 (muy separados).")


# Generate datasets from the already-created base dataset
generar_datasets_precision(df_base)


## 4. Parameterizable Dataset Generator (Punctual vs Unpunctual)

Creates datasets where you control **exactly** how many users are punctual
and how many are unpunctual, with differentiated profiles and schedules.


In [ ]:
def generar_dataset_parametrizable(
        n_total,
        n_puntuales,
        n_impuntuales,
        guardar_csv=False,
        nombre_archivo="dataset_parametrizado.csv"):
    """
    Generates a controlled dataset where the exact proportion
    of punctual and unpunctual drivers is specified.

    Profile differences:
    - Punctual: more workers, fixed early schedules (6-9h start, 14-17h end),
                high usage frequency (4-7 days/week), nearby origin
    - Unpunctual: more students/retired, variable schedules (10-18h start, 19-23h end),
                  low frequency (1-4 days/week), more distant origin

    Args:
        n_total: total number of records (must equal n_puntuales + n_impuntuales)
        n_puntuales: number of punctual drivers to generate
        n_impuntuales: number of unpunctual drivers to generate
        guardar_csv: if True, saves the result as CSV
        nombre_archivo: name of the output CSV file
    """

    if n_puntuales + n_impuntuales != n_total:
        raise ValueError("n_puntuales + n_impuntuales debe ser igual a n_total")

    datos = []

    # --- Auxiliary generation functions ---

    def seleccionar_profesion(puntual=True):
        """Punctual users tend to be workers; unpunctual ones tend to be students/retirees."""
        if puntual:
            return random.choices(["worker", "student", "retired"], weights=[0.7, 0.2, 0.1])[0]
        else:
            return random.choices(["worker", "student", "retired"], weights=[0.2, 0.4, 0.4])[0]

    def edad_segun_profesion(tipo):
        if tipo == "worker":  return random.randint(25, 55)
        elif tipo == "student": return random.randint(18, 26)
        else:                   return random.randint(60, 85)

    def distancias_segun_profesion(tipo):
        if tipo == "worker":
            return max(1, int(np.random.normal(8, 3))), max(10, int(np.random.normal(120, 40)))
        elif tipo == "student":
            return max(1, int(np.random.normal(10, 4))), max(10, int(np.random.normal(140, 50)))
        else:
            return max(1, int(np.random.normal(20, 6))), max(10, int(np.random.normal(180, 60)))

    def parametros_bateria():
        battery_capacity = random.choice([80, 100, 120, 150])
        current_charge = random.randint(0, battery_capacity)
        charge_speed = random.randint(5, 15)
        discharge_rate = random.randint(3, 10)
        return battery_capacity, current_charge, charge_speed, discharge_rate

    # --- Generate PUNCTUAL users ---
    # Predictable schedules: arrive early, leave in the afternoon, high frequency
    for i in range(1, n_puntuales + 1):
        tipo = seleccionar_profesion(puntual=True)
        edad = edad_segun_profesion(tipo)
        origin_dist, dist = distancias_segun_profesion(tipo)
        battery_capacity, current_charge, charge_speed, discharge_rate = parametros_bateria()
        h_ini = random.randint(6, 9)
        h_fin = random.randint(14, 17)
        frecuencia = random.randint(4, 7)

        datos.append({
            "id": i, "distance": dist,
            "available_time_start": h_ini, "available_time_end": h_fin,
            "current_charge": current_charge, "battery_capacity": battery_capacity,
            "charge_speed": charge_speed, "discharge_rate": discharge_rate,
            "driver_age": edad, "driver_profession": tipo,
            "use_frequency": frecuencia, "origin_distance": origin_dist,
            "puntuality_group": "punctual"
        })

    # --- Generate UNPUNCTUAL users ---
    # Late and irregular schedules, low usage frequency
    for i in range(n_puntuales + 1, n_total + 1):
        tipo = seleccionar_profesion(puntual=False)
        edad = edad_segun_profesion(tipo)
        origin_dist, dist = distancias_segun_profesion(tipo)
        battery_capacity, current_charge, charge_speed, discharge_rate = parametros_bateria()
        h_ini = random.randint(10, 18)
        h_fin = random.randint(19, 23)
        frecuencia = random.randint(1, 4)

        datos.append({
            "id": i, "distance": dist,
            "available_time_start": h_ini, "available_time_end": h_fin,
            "current_charge": current_charge, "battery_capacity": battery_capacity,
            "charge_speed": charge_speed, "discharge_rate": discharge_rate,
            "driver_age": edad, "driver_profession": tipo,
            "use_frequency": frecuencia, "origin_distance": origin_dist,
            "puntuality_group": "late"
        })

    df = pd.DataFrame(datos)

    if guardar_csv:
        df.to_csv(nombre_archivo, index=False)
        print(f"✔ Archivo generado: {nombre_archivo}")

    return df


# --- Example: 60 total users, half punctual and half unpunctual ---
df_param = generar_dataset_parametrizable(
    n_total=60, n_puntuales=30, n_impuntuales=30,
    guardar_csv=True, nombre_archivo="dataset_parametrizado.csv"
)
print(df_param.head())


## 5. Dataset Generator with Configurable Noise

Advanced version that allows injecting **soft or hard noise** to simulate
realistic scenarios where user profiles are not perfectly separable.


In [ ]:
def generate_final_dataset(total_samples=400, noise_ratio=0.15, noise_mode="hard"):
    """
    Generates a dataset with configurable noise to experiment with the robustness
    of clustering algorithms.

    Base profiles:
    - Punctual: origin_distance ≈ 5km, start ≈ 7h, end ≈ 16h,
                low car age, premium category
    - Unpunctual: origin_distance ≈ 30km, start ≈ 11h, end ≈ 21h,
                  older car, low-end category

    Noise modes:
    - 'soft': slightly shifts the values from the base profile
    - 'hard': assigns completely opposite or extreme values
              (simulates data errors or atypical users)

    Args:
        total_samples: total number of drivers to generate
        noise_ratio: fraction of records with noise (0.0 = no noise, 1.0 = all noise)
        noise_mode: 'soft' or 'hard'
    """

    data = []

    for i in range(total_samples):
        # The first half are punctual, the second half unpunctual
        is_punctual = i < (total_samples // 2)
        # Determine whether this record will have injected noise
        is_noisy = random.random() < noise_ratio

        # --- Base profile by user type ---
        if is_punctual:
            prof, o_dist, t_start, t_end, c_age, c_cat, c_cond = (
                "worker", 5, 7, 16, 2, "premium", "excellent"
            )
        else:
            prof, o_dist, t_start, t_end, c_age, c_cat, c_cond = (
                random.choice(["student", "retired"]), 30, 11, 21, 12, "low_end", "poor"
            )

        # --- Noise injection ---
        if is_noisy and noise_mode == "hard":
            # Hard noise: values completely outside the usual profile
            o_dist = random.uniform(80, 150)  # extreme distances
            t_start = random.uniform(0, 24)   # random hour
            prof = "student" if is_punctual else "worker"  # inverted profession
            c_cat = "low_end" if is_punctual else "premium"  # inverted category

        # Add Gaussian variability to the profile (with or without noise)
        data.append({
            "origin_distance": max(0, random.gauss(o_dist, 1.5)),
            "driver_profession": prof,
            "available_time_start": random.gauss(t_start, 0.5),
            "available_time_end": random.gauss(t_end, 0.5),
            "car_age": max(0, random.gauss(c_age, 1.2)),
            "car_category": c_cat,
            "car_condition": c_cond,
            "punctuality_group": "punctual" if is_punctual else "late"
        })

    df = pd.DataFrame(data)
    return df


# --- Example: 400 samples with 15% hard noise ---
df_noisy = generate_final_dataset(total_samples=400, noise_ratio=0.15, noise_mode="hard")
df_noisy.to_csv("dataset_final.csv", index=False)
print(f"✔ Dataset con ruido generado: {len(df_noisy)} registros")
print(df_noisy["punctuality_group"].value_counts())
